# Part 8: Remote Sensing Foundation Models

理解 MAE、DINO、对比学习和视觉语言预训练如何形成通用遥感表征，以及如何做线性探测与微调。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`large unlabeled corpus → self-supervised pretraining → encoder → linear probe / fine-tune / prompt adaptation`

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd().resolve()
while not (ROOT / 'scripts').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
print('repository:', ROOT)

## 核心概念

- **MAE**：遮挡大部分 patch，从可见 token 重建缺失内容，学习空间与光谱结构。
- **DINO**：教师–学生自蒸馏，不依赖人工标签形成语义表征。
- **Contrastive/VLM**：拉近匹配影像–文本或同一地点多视图，拉远不匹配样本。
- **Transfer protocol**：先 linear probe，再全量/参数高效微调，并与从头训练公平比较。

## 本地真实训练结果

长训练由脚本执行；这里读取本机已经生成的指标和产物。GitHub 中不会包含 `outputs/`。

In [ ]:
output = ROOT / r'outputs/part8_scalemae'
metrics_path = output / 'metrics.json'
print('local output:', output)
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2) if metrics_path.exists() else '尚未运行：请执行下方命令。')

## 可恢复的脚本训练

默认不在 Notebook 内启动长训练。将 `RUN_SHORT_TRAIN` 改为 `True` 可执行一次小规模 smoke run；正式命令请在终端执行。

In [ ]:
RUN_SHORT_TRAIN = False
command = r'''conda run --no-capture-output -n remote-sensing-dl python scripts/real_training/train_foundation.py --data-root data/eurosat/EuroSAT_RGB --output outputs/part8_scalemae'''
print(command)
if RUN_SHORT_TRAIN:
    import subprocess
    subprocess.run(command, cwd=ROOT, shell=True, check=True)

## 结果解释

正式结果必须包含非空 checkpoint、完整 epoch 日志、验证指标与预测图；出现缺失时先检查数据配对、CUDA 与配置，而不要直接扩大模型。

## 练习

设计一个 linear probe 与 full fine-tuning 的公平对比表，固定数据划分、增强、epoch 与评估指标。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：预训练数据与下游测试泄漏；波段/分辨率不匹配；只比较单一微调策略；忽略预训练算力和数据许可。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例